In [1]:
import psi4
import pandas as pd
import os
import numpy as np
from lps_uscf import lps_solver

In [2]:
csv_file = 'open_shell_15atoms_vs_uhf.csv'

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    print(f"-> Loaded existing results: {len(df)} rows found.")
else:
    df = pd.DataFrame()
    print("-> No existing file found. Starting fresh.")

-> Loaded existing results: 131 rows found.


In [36]:
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.166666
EXC = ['GGA_X_PBE', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 10000
DAMPING = [0.99, 0.9, 0.00009]
# D_guess = [GUESS_A, GUESS_B]
D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

mol = psi4.geometry("""
units bohr
0 3
Si
symmetry c1
""")

E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,mol,DAMPING,FA,D_guess,DIIS,verbose)
print('\nFinal SCF energy: %.6f Hartree' % E)

Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      2941.69488562    2.94169E+03    2.39654E+02
SCF Iter  2:      2873.56927670   -6.81256E+01    2.35185E+02
SCF Iter  3:      2806.79201719   -6.67773E+01    2.30797E+02
SCF Iter  4:      3708.84084848    9.02049E+02    3.39813E+02
SCF Iter  5:      4576.22860948    8.67388E+02    3.86179E+02
SCF Iter  6:      5511.80797960    9.35579E+02    4.22849E+02
SCF Iter  7:      6534.72706047    1.02292E+03    4.59503E+02
SCF Iter  8:      6416.24658169   -1.18480E+02    4.53088E+02
SCF Iter  9:      6299.37071818   -1.16876E+02    4.46784E+02
SCF Iter 10:      6188.98141998   -1.10389E+02    4.40627E+02
SCF Iter 11:      6080.59588597   -1.08386E+02    4.34732E+02
SCF Iter 12:      5973.94414341   -1.06652E+02    4.29206E+02
SCF Iter 13:      5877.81724999   -9.61269E+01    4.24064E+02
SCF Iter 14:      5782.63070633   -9.51865E+01    4.18272E+02
SCF Iter 15: 

In [37]:
GUESS_A = Da
GUESS_B = Db

In [39]:
psi4.core.set_output_file('output.dat', False)

ATOMS = {
    # Period 3 (Na-Ar)
    # 'Na': {'mult': 2},  # [Ne] 3s1
    # 'Mg': {'mult': 1},  # [Ne] 3s2
    # 'Al': {'mult': 2},  # [Ne] 3s2 3p1
    'Si': {'mult': 3},  # [Ne] 3s2 3p2 
    # 'P':  {'mult': 4},  # [Ne] 3s2 3p3 
    # 'S':  {'mult': 3},  # [Ne] 3s2 3p4
    # 'Cl': {'mult': 2},  # [Ne] 3s2 3p5
    # 'Ar': {'mult': 1},  # [Ne] 3s2 3p6
    # Period 4 (Selected)
    # 'K':  {'mult': 2},  # [Ar] 4s1
    # 'Ca': {'mult': 1},  # [Ar] 4s2
    # 'Cu': {'mult': 2},  # [Ar] 3d10 4s1 
    # 'Zn': {'mult': 1},  # [Ar] 3d10 4s2
    # 'Kr': {'mult': 1},  # [Ar] 3d10 4s2 4p6
}

METHOD = "TF0.111111W FA"
TP = ['LDA_K_TF', 1.0]
LAMBDA = 0.111111
# EXC = ['LDA_X', 1.0, 'LDA_C_VWN', 0.0]
EXC = ['GGA_X_PBE', 0.0, 'LDA_C_VWN', 0.0]
FA = [True, 1.0]
DIIS = True
MAX_ITER = 10000
DAMPING = [0.99, 0.9, 0.00002]
D_guess = [GUESS_A, GUESS_B]
# D_guess = None
verbose=True

psi4.set_options({'basis': 'UGBS_S', 
                  'DFT_SPHERICAL_POINTS': 6, 
                  'DFT_RADIAL_POINTS': 1000})

for atom in ATOMS:
    
    if not df.empty:
        exists = df[
            (df['Atom'] == atom) & 
            (df['Method'] == METHOD) & 
            (df['Basis'] == psi4.core.get_global_option("BASIS"))
        ]
        if not exists.empty:
            print(f"Skipping {atom} (Already exists for {METHOD}/{psi4.core.get_global_option("BASIS")})")
            continue

    print(f"Calculating {atom} with {METHOD}...")
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    try:
        E, Da, Db, iterations = lps_solver(MAX_ITER,TP,EXC,LAMBDA,MOL,DAMPING,FA,D_guess,DIIS,verbose)
        if iterations >= MAX_ITER:
            print("  !!! SCF failed to converge (Max cycles exceeded).")
        else:
            print(f"Calculated Energy: {E:.4f} Hartree")
            row = {
                "Method": METHOD,
                "Atom": atom,
                "Basis": psi4.core.get_global_option("BASIS"),
                "Grid_Sph": psi4.core.get_global_option("DFT_SPHERICAL_POINTS"),
                "Grid_Rad": psi4.core.get_global_option("DFT_RADIAL_POINTS"),
                "Energy,Ha": round(E, 6),
                "Iterations": iterations,
                "DIIS": DIIS,
                "Damp_Start": DAMPING[0],
                "Damp_End": DAMPING[1],
                "Damp_Cutoff": DAMPING[2]
            }
            
            df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    
    except Exception as e:
        print(f"  !!! Failed {atom}. Error: {e}")
        continue

Calculating Si with TF0.111111W FA...
Number of basis functions:   31

Starting SCF iterations:

    Iter            Energy            Delta E         dRMS

SCF Iter  1:      -299.54021904   -2.99540E+02    1.33686E+01
SCF Iter  2:      -299.66516969   -1.24951E-01    1.32038E+01
SCF Iter  3:      -299.69539611   -3.02264E-02    1.31403E+01
SCF Iter  4:      -299.62985658    6.55395E-02    1.31388E+01
SCF Iter  5:      -299.44797395    1.81883E-01    1.31454E+01
SCF Iter  6:      -299.22782467    2.20149E-01    1.31595E+01
SCF Iter  7:      -298.95770296    2.70122E-01    1.31802E+01
SCF Iter  8:      -296.16450842    2.79319E+00    1.39714E+01
SCF Iter  9:      -295.79915507    3.65353E-01    1.39465E+01
SCF Iter 10:      -293.05142263    2.74773E+00    1.36682E+01
SCF Iter 11:      -285.20748762    7.84394E+00    1.46237E+01
SCF Iter 12:      -266.83549764    1.83720E+01    1.81630E+01
SCF Iter 13:      -267.58873589   -7.53238E-01    1.78578E+01
SCF Iter 14:      -259.20222496    8.

In [40]:
df.to_csv(csv_file, index=False)
df

,Method,Atom,Basis,Grid_Sph,Grid_Rad,"Energy,Ha",Iterations,DIIS,Damp_Start,Damp_End,Damp_Cutoff
0,TFW LDA,Na,UGBS_S,6,1000,-108.912083,284,True,0.90,0.9,0.00009
1,TFW LDA,Mg,UGBS_S,6,1000,-135.553609,237,True,0.90,0.9,0.00009
2,TFW LDA,Al,UGBS_S,6,1000,-165.669941,387,True,0.90,0.9,0.00009
3,TFW LDA,Si,UGBS_S,6,1000,-199.392286,517,True,0.90,0.9,0.00009
4,TFW LDA,P,UGBS_S,6,1000,-236.833553,342,True,0.90,0.9,0.00009
...,...,...,...,...,...,...,...,...,...,...,...
131,TF0.111111W FA,S,UGBS_S,6,1000,-413.577377,8321,True,0.99,0.9,0.00009
132,TF0.111111W FA,Ca,UGBS_S,6,1000,-702.938291,5212,True,0.99,0.9,0.00009
133,TF0.111111W FA,Zn,UGBS_S,6,1000,-1842.940240,6431,True,0.99,0.9,0.00009
134,TF0.111111W FA,Kr,UGBS_S,6,1000,-2842.167575,7460,True,0.99,0.9,0.00009


In [ ]:
## TF0.111111W PBE for O and F used TF0.111111W FA densities of O and F as initial guesses
## TF0.111111W FA for Si, Kr used TF0.166666W FA initial guess

In [22]:
ATOMS = {
    'He':  {'mult': 1}, 
    'Be': {'mult': 1},
    'Ne': {'mult': 1}, 
    'Mg': {'mult': 1},
    'Ar':  {'mult': 1}, 
    'Ca':  {'mult': 1},
    'Zn':  {'mult': 1}, 
    'Kr':  {'mult': 1}
}
psi4.core.set_output_file('output.dat', False)
psi4.set_options({'basis': 'UGBS',
                  'scf_type': 'PK'})
rhf_energies = {}
rhf_homos = {}
for atom in ATOMS:
    MOL = psi4.geometry(f"0 {ATOMS[atom]['mult']}\n {atom}\nsymmetry c1")
    E, wfn = psi4.energy('SCF', return_wfn=True)
    homo = wfn.epsilon_a().np[wfn.nalpha()-1]
    rhf_energies[atom] = round(E, 6)
    rhf_homos[atom] = round(homo, 6)
    print(f"RHF/UGBS {atom} Energy: {E:.4f} Hartree, IP: {homo:.4f}")

RHF/UGBS He Energy: -2.8617 Hartree, IP: -0.9180
RHF/UGBS Be Energy: -14.5730 Hartree, IP: -0.3093
RHF/UGBS Ne Energy: -128.5471 Hartree, IP: -0.8504
RHF/UGBS Mg Energy: -199.6146 Hartree, IP: -0.2530
RHF/UGBS Ar Energy: -526.8175 Hartree, IP: -0.5910
RHF/UGBS Ca Energy: -676.7582 Hartree, IP: -0.1955
RHF/UGBS Zn Energy: -1777.8481 Hartree, IP: -0.2925
RHF/UGBS Kr Energy: -2752.0549 Hartree, IP: -0.5242


In [26]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
energy_table = df.pivot(index='Atom', columns='Method', values='Energy,Ha')
energy_table = energy_table.reindex(atom_order)
energy_table['RHF/UGBS'] = pd.Series(rhf_energies)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
energy_table = energy_table[new_order]

In [28]:
reference = energy_table['RHF/UGBS']

res = {}
for method, energies in energy_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
energy_table = pd.concat([energy_table, stats_df], sort=False)

In [29]:
display(energy_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-2.951247,-3.097069,-3.019526,-2.861680
Be,-15.040391,-15.388877,-14.660508,-14.573023
Ne,-132.508856,-133.573716,-128.291707,-128.547083
Mg,-204.540690,-205.864549,-198.305386,-199.614621
Ar,-537.208760,-539.346529,-522.928926,-526.817486
Ca,-690.396266,-692.814969,-672.808485,-676.758154
Zn,-1812.192490,-1816.067937,-1773.727670,-1777.848060
Kr,-2795.914635,-2800.696590,-2741.665221,-2752.054860
MAE(Ha),13.960000,15.970000,3.020000,NaN
rMAE(%),2.420000,3.690000,1.110000,NaN


In [33]:
atom_order = ['He', 'Be', 'Ne', 'Mg', 'Ar', 'Ca', 'Zn', 'Kr']
mu_table = df.pivot(index='Atom', columns='Method', values='ChemPot,Ha')
mu_table = mu_table.reindex(atom_order)
mu_table['RHF/UGBS'] = pd.Series(rhf_homos)
new_order = ['TFD0.166666W', 'TF0.166666W PBEx', 'TF0.166666W FA', 'RHF/UGBS']
mu_table = mu_table[new_order]

In [35]:
reference = mu_table['RHF/UGBS']

res = {}
for method, energies in mu_table.items():
    if method == 'RHF/UGBS': 
        continue 
    
    mae  = (energies - reference).abs().mean()
    rmae = (((energies - reference).abs()) / reference * -100).mean()
    res[method] = round(mae, 2), round(rmae, 2)

mae_row  = {method: values[0] for method, values in res.items()}
rmae_row = {method: values[1] for method, values in res.items()}
stats_df = pd.DataFrame([mae_row, rmae_row], index=['MAE(Ha)', 'rMAE(%)'])
mu_table = pd.concat([mu_table, stats_df], sort=False)

In [36]:
display(mu_table)

,TFD0.166666W,TF0.166666W PBEx,TF0.166666W FA,RHF/UGBS
He,-0.067106,-0.081193,-0.313483,-0.917956
Be,-0.069849,-0.081756,-0.272375,-0.309271
Ne,-0.072529,-0.082214,-0.231390,-0.850411
Mg,-0.072963,-0.082281,-0.224844,-0.253048
Ar,-0.073833,-0.082424,-0.211897,-0.590989
Ca,-0.074040,-0.082458,-0.208863,-0.195527
Zn,-0.074767,-0.082581,-0.198303,-0.292463
Kr,-0.075063,-0.082633,-0.194080,-0.524161
MAE(Ha),0.420000,0.410000,0.260000,NaN
rMAE(%),80.310000,77.800000,40.980000,NaN
